In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from glob import glob
import wandb
import itertools
from datetime import datetime
from time import time
from tqdm import tqdm

from sklearn.neighbors import KDTree

from hydra import compose, initialize
from hydra.utils import instantiate
from omegaconf import OmegaConf

from torch.utils.data import DataLoader
from pytorch_metric_learning.distances import LpDistance

from opr.utils import set_seed
from opr.testing import get_recalls
from opr.trainers.place_recognition import UnimodalPlaceRecognitionTrainer

In [3]:
import math
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple, Union

import cv2
import open3d as o3d
import gdown
import numpy as np
import pandas as pd
import torch
from loguru import logger
from omegaconf import OmegaConf
from pandas import DataFrame
from torch import Tensor
from torch.utils.data import Dataset

from opr.datasets.augmentations import (
    DefaultCloudSetTransform,
    DefaultCloudTransform,
    DefaultImageTransform,
    DefaultSemanticTransform,
)
from opr.datasets.projection import Projector
from opr.datasets.soc_utils import (
    get_points_labels_by_mask,
    instance_masks_to_objects,
    pack_objects,
    semantic_mask_to_instances,
)
from opr.optional_deps import lazy

# Lazy-load MinkowskiEngine - will return real module or helpful stub
ME = lazy("MinkowskiEngine", feature="sparse convolutions")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


# Aggregate datasets

In [4]:
DATA_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition/data/aggregated_datasets")

In [5]:
scans_dir = "/home/docker_mmpr/Datasets/2025-03-26-mmpr-datasets/keyframe-lidar-maps/keyframe-lidar-maps/mmpr_dataset/map7/keyframe_map/scans/"
poses_csv = "/home/docker_mmpr/Datasets/2025-03-26-mmpr-datasets/keyframe-lidar-maps/keyframe-lidar-maps/mmpr_dataset/map7/keyframe_map/poses.csv"

scan_files = sorted(glob(os.path.join(scans_dir, "*.pcd")))
poses = pd.read_csv(poses_csv).rename(columns={'px':'x', 'py':'y', 'pz':'z'})
df1 = poses.copy()
df1["lidar_path"] = scan_files
df1["dataset_id"] = "mmpr_map7"
df1 = df1.drop(columns='# ts')

In [ ]:
scans_dir = "/home/docker_mmpr/Datasets/iilabs_dataset/iilabs3d_dataset/benchmark/livox_mid-360/nav_a_omni/lidar0"
traj_csv = "/home/docker_mmpr/Datasets/iilabs_dataset/iilabs3d_dataset/benchmark/livox_mid-360/nav_a_omni/traj_with_lidar.csv"
traj = pd.read_csv(traj_csv)

df2 = traj[["lidar_timestamp","x","y","z","qx","qy","qz","qw"]].copy()

df2["lidar_path"] = df2["lidar_timestamp"].apply(
    lambda ts: os.path.join(scans_dir, f"{str(ts).zfill(6)}.pcd")
)
df2["dataset_id"] = "iilabs_nav_a_omni"
df2 = df2.drop(columns='lidar_timestamp')

In [7]:
def load_tiers_dataset(name):
    scans_dir = f"/home/docker_mmpr/Datasets/tiers_dataset/{name}/lidar"
    poses_csv = f"/home/docker_mmpr/Datasets/tiers_dataset/{name}/poses.csv"
    
    scan_files = sorted(glob(os.path.join(scans_dir, "*.pcd")))
    poses = pd.read_csv(poses_csv, names=['timestamp', 'x', 'y', 'z', 'qx', 'qy', 'qz', 'qw'], sep=' ')
    
    # Make sure lengths match
    min_len = min(len(scan_files), len(poses))
    poses = poses.iloc[:min_len].reset_index(drop=True)
    scan_files = scan_files[:min_len]
    
    df = poses.copy()
    df["lidar_path"] = scan_files
    df["dataset_id"] = name
    df = df[["x","y","z","qx","qy","qz","qw","lidar_path","dataset_id"]]
    
    return df

In [8]:
final_df = pd.concat(
    [
        df1, 
        df2, 
        load_tiers_dataset('indoor08'), 
        load_tiers_dataset('indoor10'), 
        load_tiers_dataset('indoor11')
    ],
    ignore_index=True
)
final_df

,x,y,z,qx,qy,qz,qw,lidar_path,dataset_id
0,0.043543,-0.028252,0.002140,-0.006958,0.002683,-0.386159,0.922402,/home/docker_mmpr/Datasets/2025-03-26-mmpr-dat...,mmpr_map7
1,0.043718,-0.024046,0.007204,-0.009612,0.006643,-0.382982,0.923682,/home/docker_mmpr/Datasets/2025-03-26-mmpr-dat...,mmpr_map7
2,0.034930,-0.019047,0.005748,-0.009378,0.007154,-0.383605,0.923422,/home/docker_mmpr/Datasets/2025-03-26-mmpr-dat...,mmpr_map7
3,0.041284,-0.020388,0.010731,-0.006994,0.009879,-0.383355,0.923522,/home/docker_mmpr/Datasets/2025-03-26-mmpr-dat...,mmpr_map7
4,0.054189,-0.029612,0.018326,0.002912,0.031283,-0.386161,0.921896,/home/docker_mmpr/Datasets/2025-03-26-mmpr-dat...,mmpr_map7
...,...,...,...,...,...,...,...,...,...
9940,2.568930,3.631894,-0.208597,-0.008437,-0.003067,-0.000117,0.999960,/home/docker_mmpr/Datasets/tiers_dataset/indoo...,indoor11
9941,2.571672,3.634535,-0.211248,-0.009032,-0.003670,-0.000087,0.999952,/home/docker_mmpr/Datasets/tiers_dataset/indoo...,indoor11
9942,2.571711,3.637547,-0.215515,-0.008623,-0.003633,-0.000070,0.999956,/home/docker_mmpr/Datasets/tiers_dataset/indoo...,indoor11
9943,2.573539,3.630533,-0.210821,-0.009050,-0.003821,-0.000553,0.999952,/home/docker_mmpr/Datasets/tiers_dataset/indoo...,indoor11


In [9]:
final_df.to_csv(DATA_ROOT / "dataset.csv", index=False)

# Dataset

In [ ]:
class UnifiedDataset(Dataset):
    csv_path: Path 
    dataset_df: DataFrame 
    pointcloud_set_transform: DefaultCloudSetTransform
    pointcloud_quantization_size: float

    def __init__(
        self,
        csv_path: Union[str, Path],
        positive_threshold: float = 10.0,
        negative_threshold: float = 50.0,
        pointcloud_quantization_size: float = 0.05
    ):
        self.csv_path = Path(csv_path)
        self.pointcloud_quantization_size = pointcloud_quantization_size

        if not self.csv_path.exists():
            raise FileNotFoundError(f"Given scans_path={self.csv_path} doesn't exist")

        self.dataset_df = pd.read_csv(self.csv_path)

        if positive_threshold < 0.0:
            raise ValueError(f"positive_threshold must be non-negative, but {positive_threshold!r} given.")
        if negative_threshold < 0.0:
            raise ValueError(f"negative_threshold must be non-negative, but {negative_threshold!r} given.")

        self.positives_index, self.nonnegative_index = self._build_indexes(
            positive_threshold, negative_threshold
        )
        self.positives_mask, self.negatives_mask = self._build_masks(positive_threshold, negative_threshold)

        self.pointcloud_set_transform = DefaultCloudSetTransform(
            train=True
        )

    def __len__(self) -> int:
        return len(self.dataset_df)

    def _build_indexes(
        self, positive_threshold: float, negative_threshold: float
    ) -> Tuple[List[Tensor], List[Tensor]]:
        """Build index of elements that satisfy a UTM distance threshold condition.

        Args:
            positive_threshold (float): The maximum UTM distance between two elements
                for them to be considered positive.
            negative_threshold (float): The maximum UTM distance between two elements
                for them to be considered non-negative.

        Returns:
            Tuple[List[Tensor], List[Tensor]]: Tuple (positive_indices, nonnegative_indices)
                of two lists of element indexes that satisfy the UTM distance threshold condition
                for each element in the dataset.
        """
        xyz = torch.tensor(
            self.dataset_df[["x", "y", "z"]].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )
        distances = torch.cdist(xyz, xyz)

        positives_mask = (distances > 0) & (distances < positive_threshold)
        nonnegatives_mask = distances < negative_threshold

        # Convert the boolean masks to index tensors
        positive_indices = [torch.nonzero(row).squeeze(dim=-1) for row in positives_mask]
        nonnegative_indices = [torch.nonzero(row).squeeze(dim=-1) for row in nonnegatives_mask]

        return positive_indices, nonnegative_indices

    def _build_masks(self, positive_threshold: float, negative_threshold: float) -> Tuple[Tensor, Tensor]:
        """Build boolean masks for dataset elements that satisfy a UTM distance threshold condition.

        Args:
            positive_threshold (float): The maximum UTM distance between two elements
                for them to be considered positive.
            negative_threshold (float): The maximum UTM distance between two elements
                for them to be considered non-negative.

        Returns:
            Tuple[Tensor, Tensor]: A tuple of two boolean masks that satisfy the UTM distance threshold
                condition for each element in the dataset. The first mask contains the indices of elements
                that satisfy the positive threshold, while the second mask contains the indices of elements
                that satisfy the negative threshold.
        """
        xyz = torch.tensor(
            self.dataset_df[["x", "y", "z"]].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )
        distances = torch.cdist(xyz, xyz)

        positives_mask = (distances > 0) & (distances < positive_threshold)
        negatives_mask = distances > negative_threshold

        return positives_mask, negatives_mask

    def _load_pc(self, idx: int) -> Tensor:
        filepath = self.dataset_df.iloc[idx]['lidar_path']
        
        pc = o3d.io.read_point_cloud(filepath)
        pc = np.asarray(pc.points)
        pc = torch.from_numpy(pc).to(torch.float32)
        
        return pc

    def __getitem__(self, idx: int) -> Dict[str, Tensor]:
        row = self.dataset_df.iloc[idx]
        data = {"idx": torch.tensor(idx, dtype=torch.int)}
        data["pose"] = torch.tensor(
            row[["x", "y", "z", "qx", "qy", "qz", "qw"]].to_numpy(dtype=np.float32)
        )
        pc = self._load_pc(idx)
        data["pointcloud_lidar_coords"] = pc 
        data["pointcloud_lidar_feats"] = torch.ones_like(pc[:, :1])

        return data
    
    def _collate_data_dict(self, data_list: List[Dict[str, Tensor]]) -> Dict[str, Tensor]:
        result: Dict[str, Tensor] = {}
        result["idxs"] = torch.stack([e["idx"] for e in data_list], dim=0)
        for data_key in data_list[0].keys():
            if data_key == "idx":
                continue 
            elif data_key == "pose":
                result["poses"] = torch.stack([e["pose"] for e in data_list], dim=0)
            elif data_key == "pointcloud_lidar_coords":
                coords_list = [e["pointcloud_lidar_coords"] for e in data_list]
                feats_list = [e["pointcloud_lidar_feats"] for e in data_list]
                n_points = [int(e.shape[0]) for e in coords_list]
                coords_tensor = torch.cat(coords_list, dim=0).unsqueeze(0)
                coords_tensor = self.pointcloud_set_transform(coords_tensor)
                coords_list = torch.split(
                    coords_tensor.squeeze(0),
                    split_size_or_sections=n_points,
                    dim=0
                )

                quantized_coords_list = []
                quantized_feats_list = []
                for coords, feats in zip(coords_list, feats_list):
                    quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                        coordinates=coords,
                        features=feats,
                        quantization_size=self.pointcloud_quantization_size
                    )
                    quantized_coords_list.append(quantized_coords)
                    quantized_feats_list.append(quantized_feats)
                
                result["pointclouds_lidar_coords"] = ME.utils.batched_coordinates(quantized_coords_list)
                result["pointclouds_lidar_feats"] = torch.cat(quantized_feats_list)
            elif data_key == "pointcloud_lidar_feats":
                continue 
            else: 
                raise ValueError(f"Unknown data key: {data_key!r}")
        return result
    
    def collate_fn(self, data_list: List[Dict[str, Tensor]]) -> Dict[str, Tensor]:
        """Pack input data list into batch.

        Args:
            data_list (List[Dict[str, Tensor]]): batch data list generated by DataLoader.

        Returns:
            Dict[str, Tensor]: dictionary of batched data.
        """
        return self._collate_data_dict(data_list)

# Training

In [11]:
with initialize(version_base=None, config_path="../libs/OpenPlaceRecognition/configs/"):
    cfg = compose(config_name="train_unimodal_minkloc3d")

print(OmegaConf.to_yaml(cfg))

wandb:
  disabled: true
  project: SeqPlaceRecognition
debug: false
device: 0
seed: 3121999
num_workers: 4
exp_name: finetune_minkloc3d_with_iilabs_dataset
epochs: 100
batch_expansion_threshold: 0.7
checkpoints_dir: /home/docker_mmpr/multimodal-place-recognition/data/finetuned_checkpoints_on_aggregated/
dataset:
  csv_path: /home/docker_mmpr/multimodal-place-recognition/data/aggregated_datasets/dataset.csv
  positive_threshold: 3.0
  negative_threshold: 10.0
  pointcloud_quantization_size: 0.05
sampler:
  _target_: opr.samplers.BatchSampler
  batch_size: 16
  batch_size_limit: 100
  batch_expansion_rate: 1.4
  max_batches: null
  positives_per_group: 2
  seed: ${seed}
  drop_last: true
model:
  _target_: opr.models.place_recognition.MinkLoc3D
  in_channels: 1
  out_channels: 256
  num_top_down: 1
  conv0_kernel_size: 5
  block: BasicBlock
  layers:
  - 1
  - 1
  - 1
  planes:
  - 32
  - 64
  - 64
  pooling: gem
loss:
  _target_: opr.losses.BatchHardTripletMarginLoss
  margin: 0.2
optim

## Init dataloaders

In [12]:
train_dataset = UnifiedDataset(
    csv_path=cfg.dataset.csv_path,
    positive_threshold=cfg.dataset.positive_threshold,
    negative_threshold=cfg.dataset.negative_threshold,
    pointcloud_quantization_size=cfg.dataset.pointcloud_quantization_size,
)

train_sampler = instantiate(cfg.sampler, dataset=train_dataset)

dataloaders = {}

dataloaders["train"] = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    collate_fn=train_dataset.collate_fn,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

## Init loss

In [13]:
loss_fn = instantiate(cfg.loss)
loss_fn

BatchHardTripletMarginLoss(
  (miner_fn): BatchHardTripletMiner(
    (distance): LpDistance()
  )
  (loss_fn): TripletMarginLoss(
    (distance): LpDistance()
    (reducer): AvgNonZeroReducer()
  )
)

## Init model

In [14]:
model = instantiate(cfg.model)

# load pretrained NCLT checkpoint
ckpt = torch.load("/home/docker_mmpr/multimodal-place-recognition/data/checkpoints/minkloc3d_nclt.pth")
model.load_state_dict(ckpt)
# model = model.to(cfg.device)

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
2025-09-09 20:29:18.570 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


<All keys matched successfully>

## Init optimizer and scheduler

In [15]:
optimizer = instantiate(cfg.optimizer, params=model.parameters())
scheduler = instantiate(cfg.scheduler, optimizer=optimizer)

In [16]:
checkpoints_dir = Path(
    Path(cfg.checkpoints_dir) / "v1"
)
checkpoints_dir

PosixPath('/home/docker_mmpr/multimodal-place-recognition/data/finetuned_checkpoints_on_aggregated/v1')

In [17]:
trainer = UnimodalPlaceRecognitionTrainer(
    checkpoints_dir=checkpoints_dir,
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_expansion_threshold=cfg.batch_expansion_threshold,
    wandb_log=False,
    device=cfg.device,
)

In [18]:
trainer.train(epochs=cfg.epochs, train_dataloader=dataloaders["train"])

2025-09-09 20:29:19.124 | INFO     | opr.trainers.place_recognition.unimodal:train:113 - =====> Epoch:   1/100:
2025-09-09 20:29:19.124 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:251 - => Train stage:
Train:   0%|          | 0/623 [00:00<?, ?it/s]

2025-09-09 20:35:39.842 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:284 - Train time: 06:20
2025-09-09 20:35:39.842 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:285 - Train stats: {'loss': 0.25940989816552373, 'avg_embedding_norm': 8.03500794752069, 'num_triplets': 15.97752808988764, 'num_non_zero_triplets': 11.735152487961477, 'non_zero_rate': 0.7346894678355352, 'max_pos_pair_dist': 0.9995301411775678, 'max_neg_pair_dist': inf, 'mean_pos_pair_dist': 0.6907993168547678, 'mean_neg_pair_dist': inf, 'min_pos_pair_dist': 0.3850353243693494, 'min_neg_pair_dist': 0.5787897024835859}
2025-09-09 20:35:39.863 | INFO     | opr.trainers.place_recognition.unimodal:train:113 - =====> Epoch:   2/100:
2025-09-09 20:35:39.864 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:251 - => Train stage:
2025-09-09 20:42:17.280 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:284 - Train time: 06:37
2025-09-09 20:42:17.280 | INFO     | o

In [19]:
# best_ckpt = torch.load(str(checkpoints_dir / "best.pth"))
# trainer.model.load_state_dict(best_ckpt["model_state_dict"])

In [20]:
# trainer.test(dataloaders["test"])